# 01 — Référentiel : législateurs & commissions

**Rôle :** construire la table d'identité des membres du Congrès US.

La clé centrale est le **BioGuideID** — un code alphanumérique unique (ex : `P000197` = Nancy Pelosi) attribué par la Bibliothèque du Congrès à chaque élu. C'est ce code qui relie entre eux les trois blocs de données :
- les PTR House (nom écrit à la main → on cherche le BioGuideID correspondant)
- les dépôts Sénat (BioGuideID fourni directement par Senate Stock Watcher)
- ce référentiel (parti, chambre, état, commissions)

**Source :** `unitedstates/congress-legislators` (domaine public, mis à jour en continu sur GitHub).

**Cellule 1 — Imports & chemins.** Ce notebook est autonome : il re-détecte la racine et les quelques dossiers dont il a besoin.

In [1]:
import requests, json
from collections import defaultdict
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
REFERENCE = PROJECT_ROOT / "data" / "reference"; REFERENCE.mkdir(parents=True, exist_ok=True)
RAW_REF   = REFERENCE / "raw"; RAW_REF.mkdir(parents=True, exist_ok=True)
REPORTS   = PROJECT_ROOT / "reports"; REPORTS.mkdir(parents=True, exist_ok=True)
USER_AGENT = "Ramify-QIS research (contact: <ton-email>)"
print("Référentiel ->", REFERENCE)

Référentiel -> /Users/lemairealice/Downloads/Jupiter/semaine 1/data/reference


**Cellule 2 — Petit téléchargeur JSON.** Réutilisable, avec cache disque (idempotent) et User-Agent honnête.

In [2]:
import yaml

BASE = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/"

def fetch_json(name: str, force: bool = False):
    # Télécharge {name}.yaml depuis congress-legislators (GitHub raw), avec cache disque.
    # Note: theunitedstates.io (ancienne URL JSON) retourne 403 — on passe au YAML source.
    path = RAW_REF / f"{name}.yaml"
    if path.exists() and not force:
        return yaml.safe_load(path.read_text())
    r = requests.get(BASE + f"{name}.yaml", headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status()
    data = yaml.safe_load(r.text)
    path.write_text(r.text)
    return data

**Cellule 3 — Téléchargement des 4 fichiers du référentiel.** Idempotent : si le fichier est déjà en cache local, on ne re-télécharge pas.

- `legislators-current` : les ~535 membres du Congrès actuel
- `legislators-historical` : tous les élus depuis 1789 (~12 000)
- `committees-current` : la liste des 49 commissions actives avec leur nom complet
- `committee-membership-current` : qui siège dans quelle commission (Congrès actuel uniquement)

In [3]:
legislators_current    = fetch_json("legislators-current")
print(f"  legislators-current    : {len(legislators_current)} membres actifs")
legislators_historical = fetch_json("legislators-historical")
print(f"  legislators-historical : {len(legislators_historical)} membres historiques (depuis 1789)")
committees_current     = fetch_json("committees-current")
print(f"  committees-current     : {len(committees_current)} commissions principales")
committee_membership   = fetch_json("committee-membership-current")
print(f"  committee-membership   : {len(committee_membership)} commissions + sous-commissions avec membres")
print()
print("Exemples de membres actuels :")
for p in legislators_current[:5]:
    name = p.get("name", {}).get("official_full", "?")
    last = p.get("terms", [{}])[-1]
    chambre = "Sénat" if last.get("type") == "sen" else "House"
    print(f"  {name:35s} | {chambre} | {last.get('party','?'):12s} | {last.get('state','?')}")

  legislators-current    : 537 membres actifs


  legislators-historical : 12230 membres historiques (depuis 1789)
  committees-current     : 49 commissions principales


  committee-membership   : 230 commissions + sous-commissions avec membres

Exemples de membres actuels :
  Maria Cantwell                      | Sénat | Democrat     | WA
  Amy Klobuchar                       | Sénat | Democrat     | MN
  Bernard Sanders                     | Sénat | Independent  | VT
  Sheldon Whitehouse                  | Sénat | Democrat     | RI
  John Barrasso                       | Sénat | Republican   | WY


**Cellule 4 — Mise à plat d'un législateur.** Chaque personne dans le YAML a une liste de mandats (`terms`). On prend le **dernier mandat** pour avoir le parti et la chambre les plus récents.

⚠️ Implication : si un élu a changé de parti au cours de sa carrière (ex : Justin Amash, Republican → Independent), seul le parti du dernier mandat est stocké. Pour les transactions d'une période antérieure au changement, ce champ sera approximatif.

In [4]:
def flatten_legislator(person: dict) -> dict:
    ids   = person.get("id", {})
    name  = person.get("name", {})
    terms = person.get("terms", [])
    last  = terms[-1] if terms else {}
    chamber = {"rep": "house", "sen": "senate"}.get(last.get("type"))
    return {
        "bioguide_id":   ids.get("bioguide"),
        "last_name":     name.get("last"),
        "first_name":    name.get("first"),
        "official_full": name.get("official_full"),
        "nickname":      name.get("nickname"),
        "party":         last.get("party"),
        "chamber":       chamber,
        "state":         last.get("state"),
        "district":      last.get("district"),
    }

rows = [flatten_legislator(p) for p in legislators_current + legislators_historical]
people = (pd.DataFrame(rows)
          .dropna(subset=["bioguide_id"])
          .drop_duplicates("bioguide_id")
          .reset_index(drop=True))
print("Législateurs uniques :", len(people))
people.head(3)

Législateurs uniques : 12767


,bioguide_id,last_name,first_name,official_full,nickname,party,chamber,state,district
0,C000127,Cantwell,Maria,Maria Cantwell,NaN,Democrat,senate,WA,NaN
1,K000367,Klobuchar,Amy,Amy Klobuchar,NaN,Democrat,senate,MN,NaN
2,S000033,Sanders,Bernard,Bernard Sanders,Bernie,Independent,senate,VT,NaN


**Cellule 5 — Variantes de nom.** Les PTR House écrivent les noms de façon inconsistante : parfois le prénom officiel, parfois le surnom, parfois les deux avec ou sans initiale. On pré-calcule toutes les formes connues pour que le matching dans le notebook 07 soit plus robuste.

Exemple : Bernard Sanders est aussi appelé "Bernie Sanders" dans certains dépôts → les deux formes sont stockées.

In [5]:
def name_variants(r) -> list:
    forms = set()
    if isinstance(r.official_full, str):
        forms.add(r.official_full)
    if isinstance(r.first_name, str) and isinstance(r.last_name, str):
        forms.add(f"{r.first_name} {r.last_name}")
    if isinstance(r.nickname, str) and isinstance(r.last_name, str):
        forms.add(f"{r.nickname} {r.last_name}")
    return sorted(forms)

people["declarant_name"] = people["official_full"].fillna(
    (people["first_name"].fillna("") + " " + people["last_name"].fillna("")).str.strip())
people["name_variants"] = people.apply(name_variants, axis=1)
people[["bioguide_id", "declarant_name", "name_variants"]].head(3)

,bioguide_id,declarant_name,name_variants
0,C000127,Maria Cantwell,[Maria Cantwell]
1,K000367,Amy Klobuchar,[Amy Klobuchar]
2,S000033,Bernard Sanders,"[Bernard Sanders, Bernie Sanders]"


**Cellule 6 — Index des commissions par identifiant.** Le fichier de membership utilise des codes courts appelés `thomas_id` (ex : `HSBA` = House Committee on Financial Services). On construit un dictionnaire `thomas_id → nom complet` pour pouvoir afficher des noms lisibles plutôt que des codes.

In [6]:
committee_name = {c["thomas_id"]: c["name"] for c in committees_current}
list(committee_name.items())[:3]

[('HSAG', 'House Committee on Agriculture'),
 ('HSAP', 'House Committee on Appropriations'),
 ('HSAS', 'House Committee on Armed Services')]

**Cellule 7 — Membre → liste de commissions.** On rattache à chaque BioGuideID ses commissions.

In [7]:
member_committees = defaultdict(set)
for thomas_id, members in committee_membership.items():
    cname = committee_name.get(thomas_id, thomas_id)
    for m in members:
        if m.get("bioguide"):
            member_committees[m["bioguide"]].add(cname)

print("Membres avec >=1 commission :", len(member_committees))

Membres avec >=1 commission : 528


**Cellule 8 — Flag commissions sensibles.** On marque les élus siégeant dans une commission Finance/Banking, Armed Services ou Intelligence — ces membres ont potentiellement accès à des informations non-publiques.

La détection se fait par mots-clés dans le nom de la commission (robuste aux variantes de noms).

> ⚠️ **Limite importante :** `committee-membership-current` couvre uniquement le **Congrès actuel** (119e). Pour les transactions de 2014–2022, l'appartenance aux commissions *au moment du trade* n'est pas connue. Le champ `committees_key_flag` sert donc de **proxy approximatif**, pas d'information historique exacte.

In [8]:
KEY_PATTERNS = ("financial", "banking", "finance", "armed services", "intelligence")

def committees_key_flag(committees: set) -> bool:
    blob = " | ".join(committees).lower()
    return any(p in blob for p in KEY_PATTERNS)

people["committee_membership"] = people["bioguide_id"].map(
    lambda b: sorted(member_committees.get(b, set())))
people["committees_key_flag"] = people["committee_membership"].map(committees_key_flag)
int(people["committees_key_flag"].sum())

197

**Cellule 9 — Sauvegarde du référentiel.** Les colonnes-listes sont sérialisées en JSON dans le CSV.

In [9]:
out = people.copy()
for col in ["name_variants", "committee_membership"]:
    out[col] = out[col].map(json.dumps)
ref_path = REFERENCE / "legislators.csv"
out.to_csv(ref_path, index=False)
print("Écrit :", ref_path, "-", len(out), "lignes")

Écrit : /Users/lemairealice/Downloads/Jupiter/semaine 1/data/reference/legislators.csv - 12767 lignes


**Cellule 9b — Aperçu de la table `legislators.csv` produite.**

In [10]:
def clean_committees(lst):
    # Garde uniquement les noms résolus (contiennent un espace = vrai nom, pas un code court)
    named = [c for c in lst if " " in c]
    if not named:
        return "—"
    return ", ".join(named[:2]) + ("…" if len(named) > 2 else "")

display_df = people[["bioguide_id", "official_full", "party", "chamber", "state",
                      "committees_key_flag"]].copy()
display_df["commissions"] = people["committee_membership"].map(clean_committees)
display_df.head(8)

,bioguide_id,official_full,party,chamber,state,committees_key_flag,commissions
0,C000127,Maria Cantwell,Democrat,senate,WA,True,"Joint Committee on Taxation, Senate Committee ..."
1,K000367,Amy Klobuchar,Democrat,senate,MN,False,"Joint Committee of Congress on the Library, Jo..."
2,S000033,Bernard Sanders,Independent,senate,VT,True,Senate Committee on Environment and Public Wor...
3,W000802,Sheldon Whitehouse,Democrat,senate,RI,True,Commission on Security and Cooperation in Euro...
4,B001261,John Barrasso,Republican,senate,WY,True,Senate Committee on Energy and Natural Resourc...
5,W000437,Roger F. Wicker,Republican,senate,MS,True,Commission on Security and Cooperation in Euro...
6,C001035,Susan M. Collins,Republican,senate,ME,True,"Senate Committee on Appropriations, Senate Com..."
7,C001056,John Cornyn,Republican,senate,TX,True,"Joint Committee on Taxation, Senate Committee ..."


**Cellule 10 — Mini-rapport de couverture.**

In [11]:
n = len(people)
n_comm = int((people["committee_membership"].map(len) > 0).sum())
n_key  = int(people["committees_key_flag"].sum())
print(f"Législateurs total  : {n}")
print(f"Avec >=1 commission : {n_comm} ({n_comm/n:.0%})  — Congrès actuel uniquement")
print(f"Sur commission clé  : {n_key}")
print()
print("⚠️  LIMITE : ces données de commission ne couvrent que le Congrès actuel (119e).")
print("   Pour les transactions 2014-2022, committee_membership est un proxy,")
print("   pas l'appartenance réelle au moment du trade.")

Législateurs total  : 12767
Avec >=1 commission : 528 (4%)  — Congrès actuel uniquement
Sur commission clé  : 197

⚠️  LIMITE : ces données de commission ne couvrent que le Congrès actuel (119e).
   Pour les transactions 2014-2022, committee_membership est un proxy,
   pas l'appartenance réelle au moment du trade.


Référentiel prêt ✅ — passez à **`02_house_index`**.